# CASMI 2026: library-search baseline

Precursor-mass candidate retrieval over the training structures, ranked by greedy cosine similarity between test spectra and each candidate's reference spectra. Generated from the team repo by `scripts/make_kaggle_notebook.py`.

## `src/data/spectra.py`

In [ ]:
import re

import numpy as np

PROTON = 1.007276

# Mass added to the neutral molecule M to obtain the singly charged ion m/z.
ADDUCT_DELTA = {
    "[M+H]+": PROTON,
    "[M+NH4]+": 18.033823,
    "[M-H2O+H]+": PROTON - 18.010565,
    "[M-2H2O+H]+": PROTON - 2 * 18.010565,
    "[M+Na]+": 22.989218,
    "[M+K]+": 38.963158,
    "[M-H]-": -PROTON,
    "[M-H2O-H]-": -PROTON - 18.010565,
    "[M+CH2O2-H]-": 44.998201,
    "[M+Cl]-": 34.969402,
}

MONO = {
    "H": 1.0078250319, "C": 12.0, "N": 14.0030740052, "O": 15.9949146221,
    "P": 30.97376151, "S": 31.97207069, "F": 18.99840320, "Cl": 34.96885271,
    "Br": 78.9183376, "I": 126.904468, "Si": 27.9769265327, "B": 11.0093055,
    "Se": 79.9165218, "Na": 22.98976966, "K": 38.9637069, "As": 74.9215964,
}
_FORMULA_RE = re.compile(r"([A-Z][a-z]?)(\d*)")


def formula_mass(formula):
    """Monoisotopic neutral mass of a molecular formula, NaN if unparseable/charged."""
    if not isinstance(formula, str) or any(c in formula for c in "+-[]()."):
        return np.nan
    total = 0.0
    for el, n in _FORMULA_RE.findall(formula):
        if el not in MONO:
            return np.nan
        total += MONO[el] * (int(n) if n else 1)
    return total


def neutral_mass(precursor_mz, adduct):
    """Neutral monoisotopic mass implied by a precursor m/z and adduct (NaN if adduct unknown)."""
    delta = ADDUCT_DELTA.get(adduct)
    return np.nan if delta is None else precursor_mz - delta


def clean_peaks(mzs, ints, precursor_mz, top_n=64, min_rel=0.01, max_above_precursor=2.0):
    """Relative-intensity floor, drop peaks above precursor, keep top-N, sqrt-scale, L2-normalise.

    Returns (mz float32, intensity float32) sorted by m/z; empty arrays if nothing survives.
    """
    mzs = np.asarray(mzs, dtype=np.float64)
    ints = np.asarray(ints, dtype=np.float64)
    if mzs.size == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    keep = ints >= min_rel * ints.max()
    if precursor_mz and np.isfinite(precursor_mz):
        keep &= mzs <= precursor_mz + max_above_precursor
    mzs, ints = mzs[keep], ints[keep]
    if mzs.size > top_n:
        idx = np.argpartition(-ints, top_n)[:top_n]
        mzs, ints = mzs[idx], ints[idx]
    order = np.argsort(mzs)
    mzs, ints = mzs[order], np.sqrt(ints[order])
    norm = np.linalg.norm(ints)
    if norm > 0:
        ints = ints / norm
    return mzs.astype(np.float32), ints.astype(np.float32)


def cosine(mz_a, int_a, mz_b, int_b, tol=0.01):
    """Greedy cosine similarity between two cleaned (L2-normalised) spectra."""
    if mz_a.size == 0 or mz_b.size == 0:
        return 0.0
    diff = np.abs(mz_a[:, None] - mz_b[None, :])
    ia, ib = np.nonzero(diff <= tol)
    if ia.size == 0:
        return 0.0
    prod = int_a[ia] * int_b[ib]
    order = np.argsort(-prod)
    used_a, used_b, score = set(), set(), 0.0
    for k in order:
        a, b = ia[k], ib[k]
        if a not in used_a and b not in used_b:
            used_a.add(a)
            used_b.add(b)
            score += prod[k]
    return float(score)


## `src/data/build_reference.py`

In [ ]:
import argparse
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


COLS = ["ingest_lib", "normalized_smiles", "inchikey14", "molecular_formula", "ionization_mode",
        "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"]


def build(train_path, out_dir, top_n=64, min_rel=0.01, min_peaks=3):
    os.makedirs(out_dir, exist_ok=True)
    pf = pq.ParquetFile(train_path)
    metas, mz_chunks, int_chunks = [], [], []
    offset, t0 = 0, time.time()
    for rg in range(pf.metadata.num_row_groups):
        tbl = pf.read_row_group(rg, columns=COLS)
        mz_list = tbl.column("ms2_mzs").to_pylist()
        in_list = tbl.column("ms2_normalized_intensities").to_pylist()
        df = tbl.drop(["ms2_mzs", "ms2_normalized_intensities"]).to_pandas()
        starts, lens = np.zeros(len(df), np.int64), np.zeros(len(df), np.int32)
        for i, (mz, it, pmz) in enumerate(zip(mz_list, in_list, df.precursor_mz.values)):
            m, s = clean_peaks(mz or [], it or [], pmz, top_n=top_n, min_rel=min_rel)
            starts[i], lens[i] = offset, m.size
            if m.size:
                mz_chunks.append(m)
                int_chunks.append(s)
                offset += m.size
        df["peak_start"], df["n_peaks"] = starts, lens
        metas.append(df[df.n_peaks >= min_peaks])
        print(f"row group {rg + 1}/{pf.metadata.num_row_groups}  {time.time() - t0:.0f}s", flush=True)
        del tbl, mz_list, in_list

    meta = pd.concat(metas, ignore_index=True)
    mz_all = np.concatenate(mz_chunks)
    int_all = np.concatenate(int_chunks)

    structs = (meta.groupby("inchikey14")
               .agg(smiles=("normalized_smiles", "first"),
                    formula=("molecular_formula", "first"),
                    n_spectra=("normalized_smiles", "size"),
                    n_libs=("ingest_lib", "nunique"))
               .reset_index())
    structs["mass"] = structs.formula.map(formula_mass)
    structs = structs.dropna(subset=["mass"]).sort_values("mass").reset_index(drop=True)
    idx = pd.Series(np.arange(len(structs)), index=structs.inchikey14)
    meta["struct_idx"] = meta.inchikey14.map(idx)
    meta = meta.dropna(subset=["struct_idx"])
    meta["struct_idx"] = meta.struct_idx.astype(np.int32)
    meta = meta[["struct_idx", "inchikey14", "ingest_lib", "ionization_mode", "adduct",
                 "precursor_mz", "peak_start", "n_peaks"]]

    meta.to_parquet(os.path.join(out_dir, "ref_meta.parquet"), index=False)
    structs.to_parquet(os.path.join(out_dir, "structures.parquet"), index=False)
    np.savez(os.path.join(out_dir, "ref_peaks.npz"), mz=mz_all, intensity=int_all)
    print(f"done: {len(meta):,} spectra, {len(structs):,} structures, {mz_all.size:,} peaks, "
          f"{time.time() - t0:.0f}s")


## `src/models/library_search.py`

In [ ]:
import os

import numpy as np
import pandas as pd


K = 25


class ReferenceLibrary:
    def __init__(self, ref_dir, exclude_libs=()):
        meta = pd.read_parquet(os.path.join(ref_dir, "ref_meta.parquet"))
        self.structs = pd.read_parquet(os.path.join(ref_dir, "structures.parquet"))
        peaks = np.load(os.path.join(ref_dir, "ref_peaks.npz"))
        self.mz, self.inten = peaks["mz"], peaks["intensity"]
        if exclude_libs:
            meta = meta[~meta.ingest_lib.isin(exclude_libs)]
        meta = meta.sort_values("struct_idx").reset_index(drop=True)
        self.meta = meta
        n = len(self.structs)
        counts = np.bincount(meta.struct_idx.values, minlength=n)
        self.spec_start = np.concatenate([[0], np.cumsum(counts)[:-1]])
        self.spec_count = counts
        # Structures with no remaining spectra are not retrievable (mimics unseen molecules).
        self.available = counts > 0
        self.masses = self.structs.mass.values
        self.prior = np.log1p(counts)
        self.mode = meta.ionization_mode.values
        self.pstart = meta.peak_start.values
        self.plen = meta.n_peaks.values

    def candidates(self, mass, ppm):
        tol = mass * ppm * 1e-6
        lo, hi = np.searchsorted(self.masses, [mass - tol, mass + tol])
        idx = np.arange(lo, hi)
        return idx[self.available[idx]]

    def spectra_of(self, s):
        a = self.spec_start[s]
        return range(a, a + self.spec_count[s])

    def peaks(self, j):
        a = self.pstart[j]
        return self.mz[a:a + self.plen[j]], self.inten[a:a + self.plen[j]]


def query_molecule(lib, spectra, ppm=15, use_cosine=True, max_ref_per_struct=50):
    """spectra: list of dicts with precursor_mz, adduct, ionization_mode, mz, intensity (cleaned).
    Returns list of (struct_idx, score) best first."""
    masses = [neutral_mass(s["precursor_mz"], s["adduct"]) for s in spectra]
    masses = [m for m in masses if np.isfinite(m)]
    if not masses:
        return []
    cands = lib.candidates(float(np.median(masses)), ppm)
    if cands.size == 0:
        return []
    scores = np.zeros(cands.size)
    if use_cosine:
        for ci, s in enumerate(cands):
            refs = list(lib.spectra_of(s))[:max_ref_per_struct]
            per_query = []
            for q in spectra:
                best = 0.0
                for j in refs:
                    if lib.mode[j] != q["ionization_mode"]:
                        continue
                    mz_r, in_r = lib.peaks(j)
                    best = max(best, cosine(q["mz"], q["intensity"], mz_r, in_r))
                per_query.append(best)
            scores[ci] = np.mean(per_query)
    # tiny prior so ties (and the mass-only baseline) prefer frequently measured structures
    total = scores + 1e-3 * lib.prior[cands]
    order = np.argsort(-total)
    return [(int(cands[i]), float(total[i])) for i in order]


def prepare_queries(df):
    """Group a test-format dataframe into {molecule_id: [spectrum dicts]} with cleaned peaks."""
    out = {}
    for row in df.itertuples(index=False):
        mz, it = clean_peaks(row.ms2_mzs, row.ms2_normalized_intensities, row.precursor_mz)
        out.setdefault(row.molecule_id, []).append(dict(
            precursor_mz=row.precursor_mz, adduct=row.adduct,
            ionization_mode=row.ionization_mode, mz=mz, intensity=it))
    return out


def predict(lib, queries, k=K, **kw):
    """Returns {molecule_id: [smiles, ...]} (up to k, best first, unique InChIKey14)."""
    smiles = lib.structs.smiles.values
    preds = {}
    for mol_id, spectra in queries.items():
        ranked = query_molecule(lib, spectra, **kw)[:k]
        preds[mol_id] = [smiles[s] for s, _ in ranked]
    return preds


def to_submission(preds, molecule_ids, fallback="CCO"):
    rows = []
    for m in molecule_ids:
        guesses = preds.get(m) or [fallback]
        rows.append({"molecule_id": m, "smiles": ";".join(guesses[:K])})
    return pd.DataFrame(rows)


## Run

In [ ]:
PPM = 15
import glob, os, time
import pandas as pd

t0 = time.time()
test_path = glob.glob("/kaggle/input/**/test.parquet", recursive=True)[0]
data_dir = os.path.dirname(test_path)
print("data dir:", data_dir)

build(os.path.join(data_dir, "train.parquet"), "/kaggle/working/ref")
lib = ReferenceLibrary("/kaggle/working/ref")
test = pd.read_parquet(test_path)
queries = prepare_queries(test)
preds = predict(lib, queries, ppm=PPM)
sub =to_submission(preds, test.molecule_id.drop_duplicates().tolist())
sub.to_csv("/kaggle/working/submission.csv", index=False)
for f in ["ref_meta.parquet", "ref_peaks.npz", "structures.parquet"]:
    os.remove(os.path.join("/kaggle/working/ref", f))
print(sub.shape, f"{time.time() - t0:.0f}s")
sub.head()
